# RQ1 v2.1 — LR vs LR+ — 20 split mới đã khóa trước

Đây là vòng xác nhận **v2.1**, được khóa sau khi đã xem toàn bộ kết quả v1 và trước khi chạy bất kỳ kết quả v2/v2.1 nào.

- Chạy đủ 20 split mới `2001–2020`; không dừng sớm theo p-value.
- Mỗi split có 1657/553/553 node và 5 model seed độc lập `12–16`.
- Hai biến thể dùng cùng split, cùng seed và cùng batch đã materialize. `LR+` chỉ thêm 12 UFE label-free.
- Kiểm định chính: Nadeau–Bengio trên 20 delta cấp split; Holm được áp dụng sau khi đủ 6 cặp.
- Đây là kiểm tra độ ổn định trên cùng 2763 node, **không phải** external/temporal holdout và **không thay thế** Bảng A so với paper.

Master protocol SHA-256: `3e63674a98e052d1770851009078adf4e2ae192a47dc9c72b84c987d202aa17c`.


In [1]:

from pathlib import Path
import csv
import gc
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

import numpy as np
import torch

MODEL_KEY = 'logistic_regression'
MODEL_LABEL = 'Logistic Regression'
VARIANTS = ["base", "plus"]
VARIANT_LABEL = {"base": 'LR', "plus": 'LR+'}
SHARED_CONFIG = {'C': 10.0, 'class_weight': None}
MODEL_SEEDS = [12, 13, 14, 15, 16]
RESPLIT_SEEDS = [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
EXPECTED_COUNTS = {"train": 1657, "validation": 553, "test": 553}
SOURCE_DATA_SHA256 = "8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8"
EDGE_STATS_DIM = 12
RELATION_TO_INDEX = {"call": 0, "trans": 1}
THRESHOLD = 0.5
BOOTSTRAP_REPETITIONS = 20_000
MINIMUM_PRACTICAL_DELTA = 0.005
ALPHA = 0.05
CODE_SHA256 = "a269f8fae98b199afb2ac02e92e3f57d0b2e4bd387e55019e1470072b401fbf9"
CONFIG = {'protocol_id': 'group23-rq1-logistic_regression-matched-20split-replication-v2.1', 'status': 'FROZEN AFTER ALL V1 RESULTS WERE OBSERVED AND BEFORE ANY V2/V2.1 RESULT', 'master_protocol_id': 'group23-rq1-all-six-matched-20split-replication-v2.1', 'master_protocol_sha256': '3e63674a98e052d1770851009078adf4e2ae192a47dc9c72b84c987d202aa17c', 'source_data_sha256': '8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8', 'model_pair': {'baseline': 'LR', 'plus': 'LR+'}, 'shared_configuration': {'C': 10.0, 'class_weight': None}, 'configuration_origin': 'frozen baseline winner selected on the paper validation split', 'implementation': {'estimator': 'StandardScaler + sklearn LogisticRegression', 'solver': 'liblinear', 'max_iter': 5000}, 'matching_rule': 'Within each pair and split, baseline and plus share the fixed configuration, train/validation/test node IDs, model seed, threshold, and materialized sampled batches/views. Plus differs only by receiving the 12-dimensional label-free UFE input/branch.', 'split_seeds': [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], 'split_method': '20 predetermined stratified random 60/20/20 splits over the same 2763 labeled EOA', 'model_seeds': [12, 13, 14, 15, 16], 'primary_metric': 'test Macro-F1 at a fixed threshold of 0.5', 'ufe': {'dimensions': 12, 'definition': 'rows/count/sum x in/out x call/trans inside each sampled batch', 'transform': 'log1p', 'uses_labels': False, 'fanout': [100, 100], 'batch_size': 512}, 'selection_and_blinding': {'hyperparameter_tuning_in_v2_1': False, 'configurations': 'frozen from v1 validation decisions / official Meta-IFD default', 'test_materialization': 'after validation decisions/checkpoints are frozen within each split', 'threshold': 0.5, 'early_termination_based_on_results': False}, 'inference': {'test': 'two-sided Nadeau-Bengio corrected resampled t-test on 20 split-level paired deltas', 'k': 20, 'df': 19, 'test_train_ratio': '553/1657', 'alpha': 0.05, 'multiplicity': 'Holm correction across the six primary model-pair p-values', 'minimum_practical_delta': 0.005, 'success_rule': 'mean delta >= 0.005, Nadeau-Bengio 95% CI lower bound > 0, and Holm-adjusted p < 0.05'}, 'reporting_commitment': {'report_all_six_pairs': True, 'report_all_twenty_splits': True, 'report_v1_and_v2_1_separately': True, 'do_not_pool_v1_and_v2_1_post_hoc': True, 'paper_comparison': 'Use the already completed author/paper split as Table A. V2.1 is Table B replication and is not a direct numerical comparison with the paper table.', 'limitations': ['random re-splits reuse the same 2763 labeled nodes', 'not an external, temporal, or unseen-dataset holdout', 'PyG neighborhood sampling may differ across sessions, but variants within each pair reuse identical materialized inputs']}, 'code_sha256': '__CODE_SHA256__', 'change_control': 'Any modification after any V2.1 result is viewed requires a new version; v1, the unexecuted v2 package, v2.1, and any later version must remain distinguishable.'}
CONFIG["code_sha256"] = CODE_SHA256
PROTOCOL_JSON = json.dumps(CONFIG, sort_keys=True, indent=2) + "\n"
PROTOCOL_SHA256 = hashlib.sha256(PROTOCOL_JSON.encode("utf-8")).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_int64(values):
    return hashlib.sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()


def write_csv(path, rows):
    assert rows
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


try:
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse
except ModuleNotFoundError:
    torch_version = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyg_lib", "torch_scatter", "torch_sparse", "-f", wheel_url])
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse

from scipy.stats import binomtest, t as student_t, wilcoxon
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

data_candidates = [p for p in Path("/kaggle/input").rglob("data.pt")
                   if "phish/processed/data.pt" in str(p).replace("\\", "/").lower()]
assert len(data_candidates) == 1, [str(p) for p in data_candidates]
data_sha256 = sha256_file(data_candidates[0])
assert data_sha256 == SOURCE_DATA_SHA256
loaded = torch.load(data_candidates[0], map_location="cpu", weights_only=False)
data = loaded[0] if isinstance(loaded, tuple) and len(loaded) == 2 else loaded
assert isinstance(data, HeteroData)
del loaded
gc.collect()
assert set(data.node_types) == {"EOA", "CA"}
assert data["EOA"].x.shape[1] == data["CA"].x.shape[1] == 14
labeled_mask = (data["EOA"].y == 0) | (data["EOA"].y == 1)
labeled_ids = torch.where(labeled_mask)[0].cpu().numpy().astype(np.int64)
labeled_y = data["EOA"].y[labeled_mask].cpu().numpy().astype(np.int64)
assert len(labeled_ids) == 2763 and int(labeled_y.sum()) == 1206
fanout_by_relation = {edge_type: [100, 100] for edge_type in data.edge_types}

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("/kaggle/working") / f"rq1_v2_1_20split_{MODEL_KEY}_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "protocol.json").write_text(PROTOCOL_JSON, encoding="utf-8")
assert sha256_file(run_dir / "protocol.json") == PROTOCOL_SHA256
print("Model pair:", VARIANT_LABEL)
print("Protocol SHA256:", PROTOCOL_SHA256)
print("Code SHA256:", CODE_SHA256)
print("Data SHA256:", data_sha256)
print("Torch/PyG:", torch.__version__, torch_geometric.__version__)
print("Output:", run_dir)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 725.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 26.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 20.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 22.5 MB/s eta 0:00:00
Model pair: {'base': 'LR', 'plus': 'LR+'}
Protocol SHA256: 1c65249b281a420cf8b30daf879d55df1b46c1978d8b555c27d92c587051f283
Code SHA256: a269f8fae98b199afb2ac02e92e3f57d0b2e4bd387e55019e1470072b401fbf9
Data SHA256: 8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8
Torch/PyG: 2.10.0+cu128 2.8.0.post1
Output: /kaggle/working/rq1_v2_1_20split_logistic_regression_20260926T004627835651Z


## Hàm mô hình, UFE và matched split


In [2]:

import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if MODEL_KEY == "lightgbm":
    try:
        from lightgbm import LGBMClassifier
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
        from lightgbm import LGBMClassifier


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def compute_ufe_by_node_type(batch):
    result = {nt: batch[nt].x.new_zeros((batch[nt].num_nodes, EDGE_STATS_DIM)) for nt in ("EOA", "CA")}
    for edge_type in batch.edge_types:
        source_type, relation, destination_type = edge_type
        store = batch[edge_type]
        if store.edge_index.numel() == 0:
            continue
        attributes = torch.nan_to_num(store.edge_attr[:, :2].to(dtype=batch[source_type].x.dtype), nan=0.0, posinf=0.0, neginf=0.0).clamp_min_(0.0)
        src, dst = store.edge_index[0], store.edge_index[1]
        ones = torch.ones(src.numel(), dtype=attributes.dtype)
        col = RELATION_TO_INDEX[relation] * 6
        result[destination_type][:, col + 0].index_add_(0, dst, ones)
        result[source_type][:, col + 1].index_add_(0, src, ones)
        result[destination_type][:, col + 2].index_add_(0, dst, attributes[:, 0])
        result[source_type][:, col + 3].index_add_(0, src, attributes[:, 0])
        result[destination_type][:, col + 4].index_add_(0, dst, attributes[:, 1])
        result[source_type][:, col + 5].index_add_(0, src, attributes[:, 1])
    return {nt: torch.log1p(v) for nt, v in result.items()}


def materialize_features(mask, sampler_seed, expected, shuffle):
    set_seed(sampler_seed)
    loader = NeighborLoader(data, input_nodes=("EOA", mask), num_neighbors=fanout_by_relation,
                            batch_size=512, shuffle=shuffle, num_workers=0)
    ids_parts, full_parts, label_parts = [], [], []
    started = time.perf_counter()
    for batch in loader:
        n = int(batch["EOA"].batch_size)
        ids = batch["EOA"].n_id[:n].clone(); base = batch["EOA"].x[:n].clone(); labels = batch["EOA"].y[:n].clone()
        assert bool(torch.all((labels == 0) | (labels == 1)))
        batch["EOA"].y[:] = -1
        if "CA" in batch.node_types and hasattr(batch["CA"], "y"): batch["CA"].y[:] = -1
        ufe = compute_ufe_by_node_type(batch)["EOA"][:n]
        full = torch.cat([base, ufe], dim=1)
        assert full.shape == (n, 26) and bool(torch.isfinite(full).all())
        ids_parts.append(ids.cpu()); full_parts.append(full.cpu()); label_parts.append(labels.cpu())
    ids = torch.cat(ids_parts).numpy().astype(np.int64, copy=False)
    full = torch.cat(full_parts).numpy().astype(np.float64, copy=True)
    labels = torch.cat(label_parts).numpy().astype(np.int64, copy=False)
    assert len(ids) == len(set(ids.tolist())) == expected
    assert set(ids.tolist()) == set(torch.where(mask)[0].tolist())
    return ids, full, labels, time.perf_counter() - started


def build_model(seed):
    if MODEL_KEY == "logistic_regression":
        return Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(
            C=SHARED_CONFIG["C"], class_weight=SHARED_CONFIG["class_weight"], solver="liblinear", max_iter=5000, random_state=seed))])
    if MODEL_KEY == "random_forest":
        return RandomForestClassifier(n_estimators=SHARED_CONFIG["n_estimators"], max_depth=SHARED_CONFIG["max_depth"],
                                      min_samples_leaf=1, class_weight=None, random_state=seed, n_jobs=-1)
    if MODEL_KEY == "lightgbm":
        return LGBMClassifier(objective="binary", n_estimators=SHARED_CONFIG["n_estimators"], learning_rate=SHARED_CONFIG["learning_rate"],
                              random_state=seed, n_jobs=-1, verbosity=-1)
    raise ValueError(MODEL_KEY)


def positive_scores(model, x):
    p = model.predict_proba(x); return p[:, int(np.where(model.classes_ == 1)[0][0])]


def metric_bundle(labels, scores, seconds=None):
    labels = np.asarray(labels, dtype=np.int64); scores = np.asarray(scores, dtype=np.float64); pred = (scores >= THRESHOLD).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0, 1]).ravel()
    out = {"pr_auc": float(average_precision_score(labels, scores)), "roc_auc": float(roc_auc_score(labels, scores)),
           "precision": float(precision_score(labels, pred, zero_division=0)), "recall": float(recall_score(labels, pred, zero_division=0)),
           "binary_f1": float(f1_score(labels, pred, zero_division=0)), "micro_f1": float(f1_score(labels, pred, average="micro")),
           "macro_f1": float(f1_score(labels, pred, average="macro")), "mcc": float(matthews_corrcoef(labels, pred)),
           "fpr": float(fp/(fp+tn)), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    if seconds is not None: out["seconds"] = float(seconds)
    return out


def run_matched_split(split_key, masks, sampler_seeds):
    folder = run_dir / split_key; folder.mkdir(parents=True, exist_ok=False)
    train_ids, train_full, y_train, train_ufe_seconds = materialize_features(masks["train"], sampler_seeds["train"], 1657, True)
    val_ids, val_full, y_val, val_ufe_seconds = materialize_features(masks["validation"], sampler_seeds["validation"], 553, True)
    assert set(train_ids).isdisjoint(val_ids)
    rows, paths = [], {}
    for variant in VARIANTS:
        x_train = train_full[:, :14] if variant == "base" else train_full
        x_val = val_full[:, :14] if variant == "base" else val_full
        for seed in MODEL_SEEDS:
            set_seed(seed); model = build_model(seed)
            started = time.perf_counter(); model.fit(x_train, y_train); fit_seconds = time.perf_counter() - started
            val_started = time.perf_counter(); val_scores = positive_scores(model, x_val); val_seconds = time.perf_counter() - val_started
            vm = metric_bundle(y_val, val_scores, val_seconds)
            path = folder / f"{variant}_seed_{seed}.joblib"; joblib.dump(model, path); paths[(variant, seed)] = path
            rows.append({"variant": variant, "seed": seed, "fit_seconds": fit_seconds,
                         **{f"validation_{k}": v for k, v in vm.items()}, "checkpoint_sha256": sha256_file(path)})
    write_csv(folder / "validation_runs.csv", rows)
    decision = {"status": "FROZEN VALIDATION DECISION (fixed shared config)", "split_key": split_key,
                "protocol_sha256": PROTOCOL_SHA256, "code_sha256": CODE_SHA256, "source_data_sha256": data_sha256,
                "shared_config": SHARED_CONFIG, "sampler_seeds": sampler_seeds,
                "train_target_ids_sha256": sha256_int64(sorted(train_ids)), "validation_target_ids_sha256": sha256_int64(sorted(val_ids)),
                "train_features_sha256": hashlib.sha256(train_full.tobytes()).hexdigest(),
                "validation_features_sha256": hashlib.sha256(val_full.tobytes()).hexdigest(),
                "checkpoint_sha256": {f"{r['variant']}_seed_{r['seed']}": r["checkpoint_sha256"] for r in rows},
                "test_features_or_labels_loaded_before_freeze": False, "test_metrics_computed_before_freeze": False}
    (folder / "frozen_validation_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")

    test_ids, test_full, y_test, test_ufe_seconds = materialize_features(masks["test"], sampler_seeds["test"], 553, False)
    assert set(test_ids).isdisjoint(train_ids) and set(test_ids).isdisjoint(val_ids)
    results, predictions, cube = [], [], {}
    for variant in VARIANTS:
        x_test = test_full[:, :14] if variant == "base" else test_full
        scores_by_seed = []
        for seed in MODEL_SEEDS:
            path = paths[(variant, seed)]; assert sha256_file(path) == decision["checkpoint_sha256"][f"{variant}_seed_{seed}"]
            model = joblib.load(path); started = time.perf_counter(); scores = positive_scores(model, x_test); seconds = time.perf_counter()-started
            tm = metric_bundle(y_test, scores, seconds); scores_by_seed.append(scores)
            vr = next(r for r in rows if r["variant"] == variant and r["seed"] == seed)
            results.append({"split_key": split_key, "variant": variant, "model": VARIANT_LABEL[variant], "seed": seed,
                            "validation_macro_f1": vr["validation_macro_f1"], "fit_seconds": vr["fit_seconds"],
                            **{f"test_{k}": v for k, v in tm.items()}})
            predictions.extend({"split_key": split_key, "variant": variant, "model": VARIANT_LABEL[variant], "seed": seed,
                                "node_id": int(i), "label": int(y), "score": float(s), "prediction": int(s >= THRESHOLD), "threshold": THRESHOLD}
                               for i, y, s in zip(test_ids, y_test, scores))
        cube[variant] = np.asarray(scores_by_seed)
    write_csv(folder / "test_results.csv", results); write_csv(folder / "test_predictions.csv", predictions)
    split_result = {"split_key": split_key, "protocol_sha256": PROTOCOL_SHA256, "code_sha256": CODE_SHA256,
                    "source_data_sha256": data_sha256, "test_target_ids_sorted_sha256": sha256_int64(sorted(test_ids)),
                    "test_features_sha256": hashlib.sha256(test_full.tobytes()).hexdigest(),
                    "base_mean_macro_f1": float(np.mean([r["test_macro_f1"] for r in results if r["variant"]=="base"])),
                    "plus_mean_macro_f1": float(np.mean([r["test_macro_f1"] for r in results if r["variant"]=="plus"])),
                    "ufe_seconds": {"train": train_ufe_seconds, "validation": val_ufe_seconds, "test": test_ufe_seconds}, "decision": decision}
    split_result["delta"] = split_result["plus_mean_macro_f1"] - split_result["base_mean_macro_f1"]
    np.savez_compressed(folder / "score_cube.npz", base=cube["base"], plus=cube["plus"], labels=y_test, test_ids=test_ids)
    split_result["score_cube_sha256"] = sha256_file(folder / "score_cube.npz")
    (folder / "split_result.json").write_text(json.dumps(split_result, indent=2), encoding="utf-8")
    return split_result, cube, y_test


def mean_macro(score_matrix, labels, sample):
    truth = labels[sample].astype(bool)
    pred = score_matrix[:, sample] >= THRESHOLD
    tp = (pred & truth).sum(1); fp = (pred & ~truth).sum(1)
    fn = (~pred & truth).sum(1); tn = (~pred & ~truth).sum(1)
    f1_pos = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(len(tp)), where=(2 * tp + fp + fn) > 0)
    f1_neg = np.divide(2 * tn, 2 * tn + fp + fn, out=np.zeros(len(tn)), where=(2 * tn + fp + fn) > 0)
    return float(np.mean((f1_pos + f1_neg) / 2))


def summarize_table_a(result_a, cube_a, labels_a):
    nodes = np.arange(len(labels_a))
    observed = mean_macro(cube_a["plus"], labels_a, nodes) - mean_macro(cube_a["base"], labels_a, nodes)
    rng = np.random.default_rng(20260925)
    boot = np.empty(BOOTSTRAP_REPETITIONS)
    for i in range(BOOTSTRAP_REPETITIONS):
        sample = rng.integers(0, len(labels_a), size=len(labels_a))
        boot[i] = mean_macro(cube_a["plus"], labels_a, sample) - mean_macro(cube_a["base"], labels_a, sample)
    ci = np.quantile(boot, [0.025, 0.975])
    p = min(1.0, 2 * min((np.sum(boot <= 0) + 1) / (BOOTSTRAP_REPETITIONS + 1),
                         (np.sum(boot >= 0) + 1) / (BOOTSTRAP_REPETITIONS + 1)))
    per_seed = []
    for k, seed in enumerate(MODEL_SEEDS):
        plus_ok = (cube_a["plus"][k] >= THRESHOLD) == labels_a
        base_ok = (cube_a["base"][k] >= THRESHOLD) == labels_a
        only_plus = int(np.sum(plus_ok & ~base_ok)); only_base = int(np.sum(~plus_ok & base_ok))
        per_seed.append({"seed": seed,
                         "plus_macro_f1": mean_macro(cube_a["plus"][[k]], labels_a, nodes),
                         "base_macro_f1": mean_macro(cube_a["base"][[k]], labels_a, nodes),
                         "only_plus_correct": only_plus, "only_base_correct": only_base,
                         "mcnemar_p": 1.0 if only_plus + only_base == 0 else float(binomtest(min(only_plus, only_base), only_plus + only_base, 0.5).pvalue)})
    return {"role": "retrospective matched comparison on the previously viewed paper split",
            "base_mean_macro_f1": result_a["base_mean_macro_f1"], "plus_mean_macro_f1": result_a["plus_mean_macro_f1"],
            "delta": observed, "bootstrap_95ci": [float(ci[0]), float(ci[1])], "bootstrap_p_two_sided": float(p),
            "seeds_plus_better": int(sum(r["plus_macro_f1"] > r["base_macro_f1"] for r in per_seed)), "per_seed": per_seed}


def summarize_table_b(split_results):
    deltas = np.asarray([r["delta"] for r in split_results])
    k = len(deltas); assert k == 20
    ratio = EXPECTED_COUNTS["test"] / EXPECTED_COUNTS["train"]
    variance = float(deltas.var(ddof=1))
    se = float(np.sqrt((1 / k + ratio) * variance))
    mean_delta = float(deltas.mean())
    if se == 0:
        t_stat, p, ci = (float("inf") if mean_delta else 0.0), (0.0 if mean_delta else 1.0), (mean_delta, mean_delta)
    else:
        t_stat = mean_delta / se
        p = float(2 * student_t.sf(abs(t_stat), df=k - 1))
        crit = float(student_t.ppf(0.975, df=k - 1))
        ci = (mean_delta - crit * se, mean_delta + crit * se)
    try:
        wp = float(wilcoxon(deltas).pvalue)
    except ValueError:
        wp = 1.0
    return {
        "role": "fixed-budget v2.1 20-split replication; same nodes, not an external holdout",
        "per_split": [{"split_key": r["split_key"], "base": r["base_mean_macro_f1"], "plus": r["plus_mean_macro_f1"], "delta": r["delta"]} for r in split_results],
        "base_mean_over_splits": float(np.mean([r["base_mean_macro_f1"] for r in split_results])),
        "plus_mean_over_splits": float(np.mean([r["plus_mean_macro_f1"] for r in split_results])),
        "mean_delta": mean_delta, "sd_delta": float(np.sqrt(variance)),
        "nadeau_bengio": {"se": se, "t": t_stat, "df": k - 1, "p_two_sided_raw": p, "ci95": [ci[0], ci[1]], "ratio_test_train": ratio},
        "descriptive_wilcoxon_p": wp, "splits_plus_better": int(np.sum(deltas > 0)), "splits_tied": int(np.sum(deltas == 0)),
        "raw_pair_rule_met": bool(mean_delta >= MINIMUM_PRACTICAL_DELTA and ci[0] > 0 and p < ALPHA),
        "global_holm_pending": True,
    }


## Chạy đủ 20 split v2.1


In [3]:

split_results=[]
for split_seed in RESPLIT_SEEDS:
    pos=np.arange(len(labeled_ids));trainval_pos,test_pos=train_test_split(pos,test_size=.20,random_state=split_seed,stratify=labeled_y)
    train_pos,val_pos=train_test_split(trainval_pos,test_size=.25,random_state=split_seed,stratify=labeled_y[trainval_pos])
    assert (len(train_pos),len(val_pos),len(test_pos)) == (1657,553,553)
    masks={}
    for name,p in (("train",train_pos),("validation",val_pos),("test",test_pos)):
        m=torch.zeros(data["EOA"].num_nodes,dtype=torch.bool);m[torch.from_numpy(labeled_ids[p])]=True;masks[name]=m
    assert not bool((masks["train"] & masks["validation"]).any() or (masks["train"] & masks["test"]).any() or (masks["validation"] & masks["test"]).any())
    seeds={"train":split_seed*10+1,"validation":split_seed*10+2,"test":split_seed*10+3}
    result,_,_=run_matched_split(f"V2_split_{split_seed}",masks,seeds);split_results.append(result)
    print(result["split_key"],result["base_mean_macro_f1"],result["plus_mean_macro_f1"],result["delta"],flush=True)

table_b=summarize_table_b(split_results)
write_csv(run_dir/"v2_20split_per_split.csv",table_b["per_split"])
final={"status":"completed fixed-budget v2.1 matched pair on all 20 predetermined splits","model_key":MODEL_KEY,"model_label":MODEL_LABEL,"variant_labels":VARIANT_LABEL,"master_protocol_sha256":CONFIG["master_protocol_sha256"],"protocol_sha256":PROTOCOL_SHA256,"code_sha256":CODE_SHA256,"protocol":CONFIG,"source_data_sha256":data_sha256,"v2_20split":table_b,"all_20_splits_completed":len(split_results)==20,"post_test_selection":False,"protocol_changed_after_results":False,"paper_comparison_in_this_notebook":False}
(run_dir/"final_summary.json").write_text(json.dumps(final,indent=2),encoding="utf-8")
shutil.make_archive(str(run_dir),"zip",run_dir)
print("V2 20-split mean:",table_b["base_mean_over_splits"],"->",table_b["plus_mean_over_splits"],"delta",table_b["mean_delta"])
print("NB:",table_b["nadeau_bengio"])
print("Raw pair rule met:",table_b["raw_pair_rule_met"],"| final Holm pending across all six pairs")
print("Output:",run_dir,"ZIP:",str(run_dir)+".zip")


V2_split_2001 0.9075035502395796 0.9632948360546927 0.055791285815113056
V2_split_2002 0.9256098561280904 0.9706358688437542 0.045026012715663755
V2_split_2003 0.8896001675650378 0.9650154994622635 0.07541533189722571
V2_split_2004 0.8788173150828305 0.9688143757069925 0.08999706062416202
V2_split_2005 0.8858953574060427 0.9632948360546927 0.07739947864864993
V2_split_2006 0.9111646586345381 0.9779570035343201 0.06679234489978203
V2_split_2007 0.9093347498753968 0.9615097791609909 0.05217502928559414
V2_split_2008 0.9075387263339071 0.9688412497969925 0.0613025234630854
V2_split_2009 0.9238112108329396 0.974392051017438 0.05058084018449838
V2_split_2010 0.905774876795638 0.9576914760914761 0.05191659929583814
V2_split_2011 0.9037954656609323 0.9556962025316456 0.05190073687071328
V2_split_2012 0.8859267408622535 0.9559538032656312 0.07002706240337764
V2_split_2013 0.91484865660004 0.9761521696582884 0.06130351305824844
V2_split_2014 0.9183854855055115 0.9779570035343201 0.0595715180288